In [ ]:
!pip install --upgrade ogb
!pip install rdkit==2024.3.6

!pip install torch torchvision torchaudio
!pip install torch-geometric

import ogb
print(ogb.__version__)




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_add_pool, JumpingKnowledge
from torch_geometric.loader import DataLoader
from ogb.lsc import PCQM4Mv2Dataset
from ogb.utils.mol import smiles2graph
from rdkit import Chem
from torch.utils.data import random_split
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import mean_absolute_error
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from torch_geometric.data import Data
import os

# Custom Data class
# just keeping it here in case I want to add more later (e.g., global features)
class MyData(Data):
    pass

# Model
class RubricGNN(nn.Module):
    def __init__(self, input_dim, edge_dim, hidden_dim=256, num_layers=7, dropout=0.2, use_virtual_node=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.use_virtual_node = use_virtual_node
        self.dropout = dropout

        # projecting node features up before GATv2
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        self.convs = nn.ModuleList([
            GATv2Conv(hidden_dim, hidden_dim, edge_dim=edge_dim)
            for _ in range(num_layers)
        ])
        self.bns = nn.ModuleList([
            nn.BatchNorm1d(hidden_dim)
            for _ in range(num_layers)
        ])

        # virtual node for global message passing
        # added once at the start and reused across the batch
        if self.use_virtual_node:
            self.virtual_node = nn.Parameter(torch.zeros(1, hidden_dim))
            nn.init.xavier_uniform_(self.virtual_node)

        # will concat all layer outputs to retain depth information
        self.jk = JumpingKnowledge(mode='cat')

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * num_layers, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, data, return_embeddings=False):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.input_proj(x)

        if self.use_virtual_node:
            batch_size = batch.max().item() + 1
            virtual_batch = self.virtual_node.repeat(batch_size, 1)
            x += virtual_batch[batch]

            virtual_edges = torch.arange(x.size(0), device=edge_index.device)
            virtual_edges = torch.stack([virtual_edges, virtual_edges], dim=0)
            edge_index = torch.cat([edge_index, virtual_edges], dim=1)
            edge_attr = torch.cat([
                edge_attr,
                torch.zeros(x.size(0), edge_attr.size(1), device=edge_attr.device)
            ], dim=0)

        # collect hidden states from each GAT layer
        # these are later combined via Jumping Knowledge
        xs = []
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            xs.append(x)

        x = self.jk(xs)
        out = global_add_pool(x, batch)

        # either return for t-SNE or predict property
        if return_embeddings:
            return out
        return self.fc(out).squeeze(-1)

# Setup
scaler = GradScaler()
criterion = nn.L1Loss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Train and Evaluate
def train(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # using AMP (autocast + GradScaler) for speed/memory on GPU
        # good for T4 and other smaller GPUs
        with autocast():
            out = model(batch)
            loss = criterion(out, batch.y.view(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds.append(model(batch).cpu())
            targets.append(batch.y.view(-1).cpu())
    return mean_absolute_error(torch.cat(targets), torch.cat(preds))

# Load and convert SMILES to graph
dataset = PCQM4Mv2Dataset(root='data/PCQM4Mv2', only_smiles=True)
graph_list = []
for i in range(2000000): #3803453 Total Samples
    smiles, label = dataset[i]
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        graph = smiles2graph(smiles)
        x = torch.tensor(graph['node_feat'], dtype=torch.float)
        edge_index = torch.tensor(graph['edge_index'], dtype=torch.long)
        edge_attr = torch.tensor(graph['edge_feat'], dtype=torch.float)
        y = torch.tensor([label], dtype=torch.float)
        data = MyData(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
        graph_list.append(data)

# Split dataset
total = len(graph_list)
train_size = int(0.8 * total)
val_size = total - train_size
train_set, val_set = random_split(graph_list, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)

# Clean up old model
if os.path.exists("best_GATv2_+_VN.pt"):
    os.remove("best_GATv2_+_VN.pt")

# Train model
best_model = RubricGNN(
    input_dim=graph_list[0].x.size(1),
    edge_dim=graph_list[0].edge_attr.size(1),
    hidden_dim=256,
    num_layers=7,
    use_virtual_node=True
).to(device)

optimizer = torch.optim.Adam(best_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

train_losses = []
val_maes = []
best_val = float('inf')

for epoch in range(1, 51):
    loss = train(best_model, train_loader, optimizer)
    val_mae = evaluate(best_model, val_loader)
    scheduler.step(val_mae)
    train_losses.append(loss)
    val_maes.append(val_mae)
    if val_mae < best_val:
        best_val = val_mae
        torch.save(best_model.state_dict(), "best_GATv2_+_VN.pt")
    print(f"Epoch {epoch:02d} | Loss: {loss:.4f} | Val MAE: {val_mae:.4f}")

# Loss Curve
plt.figure(figsize=(8, 6))
plt.plot(range(1, 51), train_losses, label='Training Loss', marker='o')
plt.plot(range(1, 51), val_maes, label='Validation MAE', marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss / MAE")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# Predictions vs True Values
  # should fall along y = x if model is doing well
def plot_predictions(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds.append(model(batch).cpu())
            targets.append(batch.y.view(-1).cpu())
    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()

    plt.figure(figsize=(8, 6))
    plt.scatter(targets, preds, label='Predictions', color='royalblue')
    plt.plot([min(targets), max(targets)], [min(targets), max(targets)], 'r--', label='Ideal (y = x)')
    plt.xlabel("True HOMO-LUMO Gap (eV)")
    plt.ylabel("Predicted HOMO-LUMO Gap (eV)")
    plt.title("Predictions vs. True Values")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_predictions(best_model, val_loader)

# t-SNE: graph-lvl embedding in 2D
  # not perfect spatially but useful for cluster separation
  # used k-means to color by cluster ID
def plot_tsne_with_clusters(model, loader, n_clusters=3):
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            features.append(model(batch, return_embeddings=True).cpu())
            labels.append(batch.y.view(-1).cpu())
    embeddings = torch.cat(features)
    tsne = TSNE(n_components=2, random_state=42).fit_transform(embeddings)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(tsne)
    cluster_ids = kmeans.labels_

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(tsne[:, 0], tsne[:, 1], c=cluster_ids, cmap='viridis')
    plt.colorbar(scatter, label="Cluster ID")
    plt.title("t-SNE Visualization of Graph Embeddings")
    plt.xlabel("t-SNE Dim 1")
    plt.ylabel("t-SNE Dim 2")
    plt.grid(True)
    plt.show()

plot_tsne_with_clusters(best_model, val_loader)

# Error by molecule size
def analyze_by_molecule_size_with_plot(model, loader):
    model.eval()
    small_maes, large_maes = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds = model(batch)
            sizes = torch.bincount(batch.batch)
            for i in range(batch.num_graphs):
                mae = F.l1_loss(preds[i], batch.y[i]).item()
                (small_maes if sizes[i] < 15 else large_maes).append(mae)

    small_avg = sum(small_maes) / len(small_maes)
    large_avg = sum(large_maes) / len(large_maes)

    plt.figure(figsize=(8, 6))
    bars = plt.bar(["Small (<15 atoms)", "Large (>=15 atoms)"], [small_avg, large_avg], color=['skyblue', 'salmon'])
    for bar, val in zip(bars, [small_avg, large_avg]):
        plt.text(bar.get_x() + bar.get_width() / 2, val + 0.005, f"{val:.4f}", ha='center', fontweight='bold')
    plt.ylabel("Average MAE (eV)")
    plt.title("Error Analysis by Molecule Size")
    plt.ylim(0, max(small_avg, large_avg) + 0.1)
    plt.grid(axis='y')
    plt.show()

analyze_by_molecule_size_with_plot(best_model, val_loader)
